# 11 — Primary models H1 / H2 / H3 (Stage 7b)

Negative-binomial regressions per HYPOTHESES.md. Cluster-robust SE by LAD. Bonferroni-adjusted p-values across the three primary tests (α = 0.05/3 = 0.0167).

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from schools_sunbeds import audit, config, stats

config.ensure_dirs()
audit.verify_manifest()
TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
ALPHA_BONF = config.ALPHA / config.BONFERRONI_K
print(f"Bonferroni-adjusted alpha for H1+H2+H3: {ALPHA_BONF:.4f}")

In [ ]:
panel = pd.read_parquet(sorted(config.DATA_PROCESSED.glob("exposure_panel_*.parquet"))[-1])
print(f"Panel: {len(panel):,} schools, columns:")
print(list(panel.columns))
panel.head()

## 0. Build a clean modelling frame

Drop schools with no IMD quintile (a tiny number of edge cases). Compute the population-weighted ridit for IMD2025.

In [ ]:
df = panel.dropna(subset=["imd_quintile"]).copy()
df["imd_quintile"] = df["imd_quintile"].astype("int64")
df["phase_simple"] = df["phase"].map(
    lambda p: "primary" if p in {"Primary", "Middle deemed primary"}
    else "secondary" if p in {"Secondary", "Middle deemed secondary", "All-through"}
    else "special"
)
df["ridit_imd"] = stats.compute_ridit(df["imd_quintile"], df.get("sum_pupil_50m"))
df["_pupil_offset"] = df["sum_pupil_50m"].clip(lower=1)  # offset must be >0
df["_buffer_offset"] = df["n_pupils_school"].fillna(1).clip(lower=1)
print(f"Modelling frame: {len(df)} rows")
print("Phase split:", df["phase_simple"].value_counts().to_dict())
print("IMD quintile counts:", df["imd_quintile"].value_counts().sort_index().to_dict())

## 1. H1 — route exposure ~ IMD quintile (cluster-robust SE by LA)

H1: tanning-salon density along walking routes is positively associated with area deprivation. Decision rule: IRR for IMD-Q1 vs Q5 > 1, 95% CI excluding 1.

In [ ]:
h1_df = df.loc[df["sum_pupil_50m"] > 0].copy()
h1_df["imd_q_cat"] = pd.Categorical(
    h1_df["imd_quintile"], categories=[5, 4, 3, 2, 1], ordered=True
)
h1 = stats.fit_nb_glm(
    h1_df,
    "sum_route_50m ~ C(imd_q_cat)",
    offset_col="_pupil_offset",
    cluster_col="la_code_dfe",
)
h1.summary_frame().round(3)

In [ ]:
irr_q1_vs_q5 = float(h1.irr.get("C(imd_q_cat)[T.1]", float("nan")))
ci = (float(h1.irr_low.get("C(imd_q_cat)[T.1]", float("nan"))),
      float(h1.irr_high.get("C(imd_q_cat)[T.1]", float("nan"))))
p = float(h1.pvalues.get("C(imd_q_cat)[T.1]", float("nan")))
print(f"H1 — IRR for IMD-Q1 (most deprived) vs Q5 (least deprived):")
print(f"  IRR  = {irr_q1_vs_q5:.2f}")
print(f"  95% CI: ({ci[0]:.2f}, {ci[1]:.2f})")
print(f"  p     = {p:.4f}  (Bonferroni cutoff {ALPHA_BONF:.4f})")
supported = (irr_q1_vs_q5 > 1) and (ci[0] > 1)
print(f"  H1 {'SUPPORTED' if supported else 'NOT supported'}")

## 2. H2 — route gradient steeper than buffer gradient

Stacked panel (school × measure type), NB with school random intercept and measure × IMD interaction. We use the simpler approach of fitting separate NB models per measure and comparing the ridit slopes via a paired bootstrap.

In [ ]:
# Buffer-based: model n_buffer_800m ~ ridit (no offset; per-school count)
h2_buffer = stats.fit_nb_glm(df, "n_buffer_800m ~ ridit_imd", cluster_col="la_code_dfe")
buffer_slope = float(h2_buffer.params["ridit_imd"])
buffer_rii = float(np.exp(-buffer_slope))  # ridit=0 (most deprived) / ridit=1 (least deprived)
print("Buffer-based (n_buffer_800m ~ ridit):")
print(h2_buffer.summary_frame().round(3))
print(f"  RII (Q1/Q5 IRR via ridit) = {buffer_rii:.3f}")

In [ ]:
# Route-based: model sum_route_50m ~ ridit + offset(log(sum_pupil_50m))
h2_route = stats.fit_nb_glm(
    h1_df,
    "sum_route_50m ~ ridit_imd",
    offset_col="_pupil_offset",
    cluster_col="la_code_dfe",
)
route_slope = float(h2_route.params["ridit_imd"])
route_rii = float(np.exp(-route_slope))
print("Route-based (sum_route_50m ~ ridit, offset=log(sum_pupil_50m)):")
print(h2_route.summary_frame().round(3))
print(f"  RII (Q1/Q5 IRR via ridit) = {route_rii:.3f}")

In [ ]:
rii_ratio_point = route_rii / buffer_rii
print(f"H2 headline: RII_route / RII_buffer = {rii_ratio_point:.3f}")
print(f"  > 1 means route exposure shows a steeper gradient than buffer (H2 supported)")
print(f"  < 1 means the opposite")
print("\nFormal CI for the headline ratio is computed in notebook 12 via bootstrap.")

## 3. H3 — secondary > primary exposure (phase × IMD interaction)

In [ ]:
h3_df = h1_df.loc[h1_df["phase_simple"].isin(["primary", "secondary"])].copy()
h3 = stats.fit_nb_glm(
    h3_df,
    "sum_route_50m ~ C(phase_simple) * ridit_imd",
    offset_col="_pupil_offset",
    cluster_col="la_code_dfe",
)
h3.summary_frame().round(3)

In [ ]:
phase_main = float(h3.params.get("C(phase_simple)[T.secondary]", 0.0))
phase_p = float(h3.pvalues.get("C(phase_simple)[T.secondary]", float("nan")))
irr_phase = float(np.exp(phase_main))
print(f"H3 — secondary main effect (vs primary) on per-pupil route exposure:")
print(f"  IRR(secondary / primary) = {irr_phase:.2f}")
print(f"  p = {phase_p:.4f}  (Bonferroni cutoff {ALPHA_BONF:.4f})")

## 4. Persist regression tables

In [ ]:
h1.summary_frame().to_csv(config.OUTPUTS_TABLES / f"T3a_h1_nb_route_imd_{TODAY}.csv")
h2_buffer.summary_frame().to_csv(config.OUTPUTS_TABLES / f"T3b_h2_buffer_{TODAY}.csv")
h2_route.summary_frame().to_csv(config.OUTPUTS_TABLES / f"T3c_h2_route_{TODAY}.csv")
h3.summary_frame().to_csv(config.OUTPUTS_TABLES / f"T3d_h3_phase_imd_{TODAY}.csv")
print("Wrote 4 regression tables to outputs/tables/")